# Stage 1 — MediaPipe-only baseline (no visual encoder)

Per the iterative-ablation plan: measures how much of WiTA is recoverable from the literal hand-trajectory signal alone. This is the representational upper bound for 'trajectory tells the whole story.'

**Pipeline**: MediaPipe Tasks → 21 joints × (x,y,z) + velocity + acceleration + visibility = **190 dims/frame** → resample to T_native=32 → Conformer-4 → ConvTranspose1d upsample → CTC.

**Sanity controls** built in:
- Subject-disjoint split (saved manifest)
- T_out ≥ 2L+1 assertion every batch
- Length-bucketed CER, blank prob, entropy, KL diagnostics every epoch
- Single-clip overfit and shuffled-label controls

**Honest prior**: CER 0.30-0.55. If higher, the signal extraction or model is undersized.

## Cell 1 — Install + clone

In [ ]:
%%capture
!pip install editdistance huggingface_hub hgtk 'mediapipe>=0.10.0' --quiet

import sys, os
!rm -rf /kaggle/working/wita_v2
!git clone -b iterative-ablation "https://github.com/Gaurs86/WiTA-v2.git" '/kaggle/working/wita_v2'
sys.path.insert(0, '/kaggle/working')

## Cell 2 — Config (Stage 0 + Stage 1 settings)

In [ ]:
import os, logging, random, json
import numpy as np
import torch

os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
os.makedirs('/kaggle/working/logs',        exist_ok=True)
os.makedirs('/kaggle/working/manifests',   exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-7s  %(name)s \u2014 %(message)s',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler('/kaggle/working/logs/stage1.log'),
    ]
)

from wita_v2.configs.default import (
    Config, DataConfig, AugConfig, EncoderConfig,
    RecurrentConfig, AttnDecoderConfig, TrainConfig,
)

SKELETON_CACHE = '/kaggle/working/skeleton_features_t32.pt'
SPLIT_MANIFEST = '/kaggle/working/manifests/subject_split.json'

# Stage-1 hyperparameters \u2014 v2 with augmentation + regularization
# Prior run hit 0.71 val CER but train_nll \u2192 0.0005 (pure overfit).
# Train\u2013val NLL gap was +8.05 nats. These changes target that gap directly.
T_NATIVE        = 32
UPSAMPLE        = 2          # T_out = 64
D_MODEL         = 256
N_LAYERS        = 4
N_HEADS         = 4
CONV_KERNEL     = 15
DROPOUT         = 0.2        # \u2191 from 0.1 (anti-overfit)
BATCH_SIZE      = 32
LR_PEAK         = 5e-4
WEIGHT_DECAY    = 5e-2       # \u2191 from 1e-2 (anti-overfit)
GRAD_CLIP       = 1.0
NUM_EPOCHS      = 80         # \u2193 from 200 (best epoch was 175 but plateau started ~epoch 20)
WARMUP_PCT      = 0.05
SEED            = 42

# Augmentation defaults (match plan: temporal warp \u00b115%, jitter \u00b12%, crop 60\u2013100%)
USE_AUGMENT     = True

cfg = Config(
    data=DataConfig(
        hf_repo_id='yewon816/WiTA',
        lang='english',
        max_zips=None,
        max_frames=64,
        train_split=0.90,
        seed=SEED,
    ),
    encoder=EncoderConfig(arch='siglip'),    # placeholder, unused
    train=TrainConfig(
        num_epochs=NUM_EPOCHS,
        batch_size=BATCH_SIZE,
        lr=LR_PEAK,
        weight_decay=WEIGHT_DECAY,
        grad_clip=GRAD_CLIP,
        num_workers=2,
        warmup_pct=WARMUP_PCT,
        seed=SEED,
        checkpoint_dir='/kaggle/working/checkpoints',
    ),
).build()

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True

print(f'Device     : {cfg.device}')
print(f'GPU(s)     : {torch.cuda.device_count()}')
print(f'Vocab size : {cfg.vocab.ctc_vocab_size}  (blank=0, chars={len(cfg.vocab.chars)})')
print(f'T_native   : {T_NATIVE}   T_out = {T_NATIVE * UPSAMPLE}')
print(f'Regulariz. : dropout={DROPOUT}  weight_decay={WEIGHT_DECAY}  epochs={NUM_EPOCHS}')
print(f'Augmentation: {USE_AUGMENT}')


## Cell 3 — Stream + index with subject tracking

In [ ]:
from wita_v2.datasets.subject_splits import stream_and_index_with_subjects
samples = stream_and_index_with_subjects(cfg)
print(f'Total: {len(samples)} clips across {len({s for _,_,s in samples})} subjects')

## Cell 4 — Build subject-disjoint split

In [ ]:
from wita_v2.datasets.subject_splits import build_subject_split, save_split_manifest

train_idx, val_idx, manifest = build_subject_split(
    samples, train_ratio=0.90, seed=SEED,
)
save_split_manifest(manifest, SPLIT_MANIFEST)
print(f'Train subjects: {manifest["n_subjects_train"]}   clips: {manifest["n_train_clips"]}')
print(f'Val   subjects: {manifest["n_subjects_val"]}     clips: {manifest["n_val_clips"]}')
print()
print('First 10 val subjects (held out from training):')
print(manifest['val_subjects'][:10])

## Cell 5 — Extract skeleton features (ONCE)

Per-clip pipeline: PIL frames → MediaPipe → 21 (x,y,z) → resample to T_native=32 → concat (pos, velocity, accel, visibility) → [32, 190].

In [ ]:
from wita_v2.datasets.skeleton_cache import extract_skeleton_features

if os.path.exists(SKELETON_CACHE):
    print(f'Cache exists: {SKELETON_CACHE}  ({os.path.getsize(SKELETON_CACHE)/1e6:.1f} MB)')
else:
    extract_skeleton_features(
        samples, out_path=SKELETON_CACHE,
        T_native=T_NATIVE, dtype=torch.float16,
    )

## Cell 6 — Load cache + build dataloaders with subject split

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from wita_v2.datasets.vocab import make_converter
from wita_v2.datasets.subject_splits import apply_split_from_manifest, load_split_manifest
from wita_v2.datasets.skeleton_augment import LandmarkAugment

cache = torch.load(SKELETON_CACHE, map_location='cpu', weights_only=False)
manifest = load_split_manifest(SPLIT_MANIFEST)
converter = make_converter(cfg.data.lang)

# Subject sanity
subject_per_idx = cache['subjects']
unique_subjects = sorted(set(subject_per_idx))
print(f'Cache subjects ({len(unique_subjects)}): {unique_subjects[:20]}'
      + ('...' if len(unique_subjects) > 20 else ''))
if unique_subjects == ['UNKNOWN']:
    raise RuntimeError('Cache has no subject IDs. Re-run Cell 3/5 to rebuild.')

train_set, val_set = set(manifest['train_subjects']), set(manifest['val_subjects'])
train_idx_cache = [i for i, s in enumerate(subject_per_idx) if s in train_set]
val_idx_cache   = [i for i, s in enumerate(subject_per_idx) if s in val_set]

print(f'Cache: {len(cache["feats"])} clips, {len(unique_subjects)} subjects')
print(f'Train: {len(train_idx_cache)} clips  ({len(train_set)} subjects)')
print(f'Val  : {len(val_idx_cache)} clips  ({len(val_set)} subjects)')
print(f'Frame detect rate during extraction: {cache.get("frame_detect_rate", 0)*100:.1f}%')

if len(val_idx_cache) == 0:
    raise RuntimeError('VAL SET IS EMPTY. Set max_zips=None or >=6 in Cell 2.')
if len(train_idx_cache) < cfg.train.batch_size:
    raise RuntimeError(f'Train set too small ({len(train_idx_cache)} < batch_size={cfg.train.batch_size}).')

class SkeletonDataset(Dataset):
    def __init__(self, cache, indices, converter, transform=None):
        self.cache = cache; self.indices = indices
        self.converter = converter; self.transform = transform
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        idx = self.indices[i]
        feats = self.cache['feats'][idx].float()           # [T, 190]
        if self.transform is not None:
            feats = self.transform(feats)
        label_str = self.cache['labels'][idx]
        enc, _ = self.converter.encode(label_str)
        return feats, enc

def collate(batch, pad_idx):
    feats, labels = zip(*batch)
    feats_pad  = pad_sequence(feats, batch_first=True, padding_value=0.0)
    labels_pad = pad_sequence(labels, batch_first=True, padding_value=pad_idx)
    input_lens = torch.LongTensor([f.shape[0] for f in feats])
    label_lens = torch.LongTensor([l.shape[0] for l in labels])
    return feats_pad, labels_pad, input_lens, label_lens

pad_idx = cfg.vocab.pad_idx
collate_fn = lambda b: collate(b, pad_idx=pad_idx)

# Augmentation \u2014 train only, NOT val.
train_aug = LandmarkAugment() if USE_AUGMENT else None
val_aug   = None
print(f'Train augment: {train_aug}')
print(f'Val   augment: {val_aug}')

train_ds = SkeletonDataset(cache, train_idx_cache, converter, transform=train_aug)
val_ds   = SkeletonDataset(cache, val_idx_cache,   converter, transform=val_aug)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, collate_fn=collate_fn, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, collate_fn=collate_fn)

print(f'Train batches: {len(train_loader)}   Val batches: {len(val_loader)}')
if len(train_loader) == 0:
    raise RuntimeError(f'Train loader has 0 batches.')

# Sanity check one batch
feats, labels, in_lens, lab_lens = next(iter(train_loader))
print(f'feats: {feats.shape}  labels: {labels.shape}  in_lens[:4]={in_lens[:4].tolist()}  lab_lens[:4]={lab_lens[:4].tolist()}')


## Cell 7 — Build ConformerCTC + CTC feasibility check

In [ ]:
from wita_v2.models.conformer_ctc import ConformerCTC
from wita_v2.training.diagnostics import assert_ctc_feasible

model = ConformerCTC(
    input_dim   = cache['out_dim'],
    vocab_size  = cfg.vocab.ctc_vocab_size,
    d_model     = D_MODEL,
    n_layers    = N_LAYERS,
    n_heads     = N_HEADS,
    conv_kernel = CONV_KERNEL,
    dropout     = DROPOUT,
    upsample    = UPSAMPLE,
).to(cfg.device)
print(f'Trainable params: {model.num_params:,}')

with torch.no_grad():
    lp, enc_lens = model(feats.to(cfg.device), in_lens.to(cfg.device))
print(f'log_probs: {lp.shape}   enc_lens[:4]={enc_lens[:4].tolist()}')

stats = assert_ctc_feasible(enc_lens.cpu(), lab_lens, raise_on_fail=False)
print(f'CTC feasibility this batch: feasible={stats["feasible"]}/{stats["total"]}  infeasible={stats["infeasible"]}')

import torch.nn as nn
ctc = nn.CTCLoss(blank=cfg.vocab.blank_idx, zero_infinity=True, reduction='mean')
with torch.no_grad():
    loss = ctc(lp.transpose(0,1).float(), labels.to(cfg.device), enc_lens, lab_lens.to(cfg.device))
print(f'Initial CTC loss: {loss.item():.4f}')


## Cell 8 — Single-clip overfit sanity (~30 sec)

If the model can't overfit a single clip to zero loss in 500 steps, the head is broken before any data discussion.

In [ ]:
import copy
model_overfit = copy.deepcopy(model)
opt = torch.optim.AdamW(model_overfit.parameters(), lr=1e-3)
ctc_loss_fn = nn.CTCLoss(blank=cfg.vocab.blank_idx, zero_infinity=True, reduction='mean')

# One short and one long clip (if available)
short_idx = None; long_idx = None
for i in train_idx_cache[:500]:
    L = len(cache['labels'][i])
    if L <= 4 and short_idx is None: short_idx = i
    if L >= 9 and long_idx is None:  long_idx = i
    if short_idx is not None and long_idx is not None: break

for tag, idx in [('SHORT', short_idx), ('LONG', long_idx)]:
    if idx is None:
        print(f'{tag}: no example found, skipping'); continue
    f = cache['feats'][idx].float().unsqueeze(0).to(cfg.device)
    enc, _ = converter.encode(cache['labels'][idx])
    lab = enc.unsqueeze(0).to(cfg.device)
    lab_len = torch.LongTensor([enc.shape[0]]).to(cfg.device)
    in_len  = torch.LongTensor([f.shape[1]]).to(cfg.device)
    losses = []
    for step in range(500):
        lp, el = model_overfit(f, in_len)
        loss = ctc_loss_fn(lp.transpose(0,1).float(), lab, el, lab_len)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
    print(f'{tag} ({cache["labels"][idx]!r} L={lab_lens[0] if False else len(cache["labels"][idx])}): '
          f'loss step0={losses[0]:.4f}  step100={losses[99]:.4f}  step499={losses[499]:.4f}')
del model_overfit, opt

## Cell 9 — Train Stage 1 + epoch-by-epoch diagnostics

In [ ]:
import time, math
import editdistance
from wita_v2.training.diagnostics import (
    full_diagnostic_snapshot, format_snapshot_line, assert_ctc_feasible,
)

device = cfg.device
ctc = nn.CTCLoss(blank=cfg.vocab.blank_idx, zero_infinity=True, reduction='mean')
optimizer = torch.optim.AdamW(model.parameters(), lr=LR_PEAK,
                              weight_decay=WEIGHT_DECAY, betas=(0.9, 0.999))

total_steps = NUM_EPOCHS * len(train_loader)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR_PEAK, total_steps=total_steps,
    pct_start=WARMUP_PCT, anneal_strategy='cos',
)

def decode_argmax(log_probs, enc_lens, blank):
    out = []
    argmax = log_probs.argmax(dim=-1)
    for b in range(argmax.shape[0]):
        seq = argmax[b, :int(enc_lens[b].item())].tolist()
        merged = []; prev = None
        for t in seq:
            if t != prev and t != blank: merged.append(t)
            prev = t
        out.append(merged)
    return out

best_cer = float('inf')
history = []

for epoch in range(NUM_EPOCHS):
    # ── train ──────────────────────────────────────────────────────────
    model.train()
    sum_loss = 0.0; n_batches = 0; t0 = time.time()
    for feats, labels, in_lens, lab_lens in train_loader:
        feats = feats.to(device); labels = labels.to(device)
        in_lens = in_lens.to(device); lab_lens = lab_lens.to(device)

        lp, enc_lens = model(feats, in_lens)
        # CTC feasibility — hard assertion (Stage 0 requirement)
        assert_ctc_feasible(enc_lens.cpu(), lab_lens.cpu(), raise_on_fail=True)
        loss = ctc(lp.transpose(0,1).float(), labels, enc_lens, lab_lens)
        optimizer.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step(); scheduler.step()

        sum_loss += loss.item(); n_batches += 1
    train_loss = sum_loss / max(n_batches, 1)

    # ── val ────────────────────────────────────────────────────────────
    model.eval()
    pairs = []; sum_val_loss = 0.0; n_val_batches = 0
    last_lp = last_lens = None
    with torch.no_grad():
        for feats, labels, in_lens, lab_lens in val_loader:
            feats = feats.to(device); labels = labels.to(device)
            in_lens = in_lens.to(device); lab_lens = lab_lens.to(device)
            lp, enc_lens = model(feats, in_lens)
            v_loss = ctc(lp.transpose(0,1).float(), labels, enc_lens, lab_lens)
            sum_val_loss += v_loss.item(); n_val_batches += 1
            for b, pred in enumerate(decode_argmax(lp, enc_lens, cfg.vocab.blank_idx)):
                gt = converter.decode(labels[b, :lab_lens[b].item()].int().cpu(),
                                       torch.IntTensor([lab_lens[b].item()]))
                pred_str = ''.join(cfg.vocab.chars[t-1] if 1<=t<=len(cfg.vocab.chars) else '?' for t in pred)
                pairs.append((gt, pred_str))
            last_lp = lp; last_lens = enc_lens
    val_loss = sum_val_loss / max(n_val_batches, 1)

    # ── diagnostics ────────────────────────────────────────────────────
    snap = full_diagnostic_snapshot(
        pairs=pairs, log_probs=last_lp, lengths=last_lens,
        chars=cfg.vocab.chars, train_loss=train_loss, val_loss=val_loss,
        blank=cfg.vocab.blank_idx,
    )
    history.append({'epoch': epoch+1, **{k: v for k, v in snap.items() if not isinstance(v, (list, dict))}})
    dt = time.time() - t0
    print(f'[Ep {epoch+1:3d}/{NUM_EPOCHS}] train={train_loss:.4f}  val_loss={val_loss:.4f}  '
          f'{format_snapshot_line(snap)}  {dt:.0f}s', flush=True)

    # ── checkpoint ─────────────────────────────────────────────────────
    if snap['val_cer_overall'] < best_cer:
        best_cer = snap['val_cer_overall']
        torch.save({
            'model_state_dict': model.state_dict(),
            'epoch': epoch, 'val_cer': best_cer, 'snapshot': snap,
        }, '/kaggle/working/checkpoints/stage1_best.pt')
        print(f'  \u2605 new best CER={best_cer:.4f}')

print(f'\nStage 1 done. Best val CER: {best_cer:.4f}')

# Save history
with open('/kaggle/working/logs/stage1_history.json', 'w') as f:
    json.dump(history, f, indent=2)
print('History saved.')

## Cell 10 — Final report

Pass/fail criteria from the plan:
- **CER ≤ 0.40** → trajectory carries dominant signal. Proceed to Stage 2.
- **0.40 < CER ≤ 0.65** → trajectory carries half the signal.
- **CER > 0.65** → MediaPipe extraction is too noisy or model is undersized. Investigate before proceeding.

In [ ]:
ckpt = torch.load('/kaggle/working/checkpoints/stage1_best.pt', map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
snap = ckpt['snapshot']

print(f'\n=== STAGE 1 RESULT ===')
print(f'Best val CER: {snap["val_cer_overall"]:.4f}')
print(f'\nLength-bucketed CER:')
for bucket, m in snap['length_buckets'].items():
    print(f'  L={bucket:5s}  n={m["n_clips"]:4d}  CER={m["cer"]:.4f}  '
          f'mean_pred_len={m["mean_pred_len"]:.1f}  mean_target_len={m["mean_target_len"]:.1f}')
print(f'\nEdit decomposition: {snap["edit_decomposition"]}')
print(f'Mean blank prob: {snap.get("mean_blank_prob", float("nan")):.3f}')
print(f'Mean posterior entropy (nats): {snap.get("mean_entropy_nats", float("nan")):.3f}')
print(f'KL(pred || label): {snap["kl_pred_vs_label_nats"]:.3f}')
print(f'NLL gap (val - train): {snap.get("nll_gap", float("nan")):+.4f}')
print()
if snap['val_cer_overall'] <= 0.40:
    print('\u2705 PASS: trajectory-only model breaks 0.40.  Proceed to Stage 2 (DINOv2).')
elif snap['val_cer_overall'] <= 0.65:
    print('\u26a0  PARTIAL: trajectory carries half the signal.  Visual features will add value.')
else:
    print('\u274c FAIL: investigate. Check landmark detect rate, model size, T_native.')